# Analisis Reproduktif Sistem Rekomendasi SPKLU Sulawesi

**Pendamping analisis untuk penulisan jurnal — kandidat aplikasi versi 0.14.0**

Notebook ini menjelaskan dan memverifikasi empat bagian utama sistem:

1. karakteristik serta konsolidasi dataset SPKLU Sulawesi;
2. kompatibilitas konektor dan jaringan pengisian publik/dealer;
3. model energi dan Dynamic Programming (DP) dengan state `(node, SOC)`; dan
4. hasil enam skenario baseline serta tujuh skenario sensitivitas.

> Seluruh analisis dalam notebook ini bersifat **offline**. Tidak ada pemanggilan Google Maps API, sehingga menjalankan semua cell tidak memakai kuota atau billing Google Cloud.

## Cara membaca hasil dan batas interpretasi

- Dataset merupakan snapshot lokasi, konektor, dan koordinat; status operasional charger secara real-time tidak tersedia.
- Penanda Hyundai, Wuling, dan Toyota/Lexus dipakai sebagai **jaringan dealer bersyarat**, bukan jaminan akses. Pengguna tetap perlu mengonfirmasi kebijakan operator.
- Eksperimen penelitian menggunakan konektor **CCS2**. Dukungan konektor lain tersedia di aplikasi, tetapi tidak boleh dianggap sudah dievaluasi oleh baseline CCS2.
- `graph_disconnected` berarti tidak terbentuk jalur pada graf di bawah parameter, konektor, kandidat koridor, dan data jalan eksperimen. Istilah ini bukan bukti bahwa perjalanan secara fisik mustahil dilakukan.
- Waktu dalam hasil adalah waktu berkendara dari Google Routes. **Waktu pengisian tidak dihitung**.
- Runtime mencakup latensi jaringan saat eksperimen live. Gunakan statistik state/transisi DP untuk membahas beban algoritmik dengan lebih hati-hati.

In [ ]:
from __future__ import annotations

import hashlib
import json
import sys
from html import escape
from pathlib import Path

import numpy as np
import pandas as pd

try:
    from IPython.display import HTML, display
except ImportError:  # Memungkinkan verifikasi cell tanpa Jupyter/IPython.
    class HTML:
        def __init__(self, data):
            self.data = data

    def display(value):
        if isinstance(value, HTML):
            print('[Visualisasi SVG siap ditampilkan di Jupyter]')
        else:
            print(value)

pd.set_option('display.max_columns', 60)
pd.set_option('display.max_colwidth', 80)
pd.set_option('display.width', 160)

def find_project_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / 'release_manifest.json').is_file() and (candidate / 'dataset_spklu_sulawesi.csv').is_file():
            return candidate
    raise FileNotFoundError(
        'Root proyek tidak ditemukan. Jalankan notebook dari dalam repository spklu-sulawesi.'
    )

PROJECT_ROOT = find_project_root(Path.cwd().resolve())
NOTEBOOK_DATA = PROJECT_ROOT / 'notebooks' / 'data'
DATASET_PATH = PROJECT_ROOT / 'dataset_spklu_sulawesi.csv'
MANIFEST_PATH = PROJECT_ROOT / 'release_manifest.json'
EXPORT_FIGURES = False
FIGURE_DIR = PROJECT_ROOT / 'notebooks' / 'figures'

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from app.services.dataset import (
    CHARGING_NETWORK_LABELS,
    CHARGING_NETWORK_ORDER,
    CONNECTOR_ORDER,
    load_station_catalog,
    node_is_eligible,
)
from app.services.energy import EnergyParameters, SocDiscretizer

print(f'Root proyek: {PROJECT_ROOT}')
print('Mode analisis: OFFLINE — tanpa request Google Maps API')

In [ ]:
PALETTE = ['#0f766e', '#2563eb', '#d97706', '#dc2626', '#7c3aed', '#0891b2']

def show_table(frame: pd.DataFrame, decimals: int = 3):
    display(frame.round(decimals).reset_index(drop=True))

def show_svg(svg: str, filename: str | None = None):
    if EXPORT_FIGURES and filename:
        FIGURE_DIR.mkdir(parents=True, exist_ok=True)
        (FIGURE_DIR / filename).write_text(svg, encoding='utf-8')
    display(HTML(svg))

def horizontal_bar_svg(labels, values, title, value_label='', color='#0f766e'):
    labels = [str(label) for label in labels]
    values = [float(value) for value in values]
    width, left, right = 900, 255, 85
    row_height, top, bottom = 38, 58, 44
    height = top + bottom + row_height * len(labels)
    plot_width = width - left - right
    maximum = max(values) if values and max(values) > 0 else 1.0
    pieces = [
        f'<svg viewBox="0 0 {width} {height}" xmlns="http://www.w3.org/2000/svg" role="img" aria-label="{escape(title)}">',
        '<style>text{font-family:Arial,sans-serif;fill:#173042}.title{font-size:20px;font-weight:700}.label{font-size:14px}.value{font-size:13px;font-weight:700}</style>',
        f'<text class="title" x="20" y="30">{escape(title)}</text>',
    ]
    for index, (label, value) in enumerate(zip(labels, values)):
        y = top + index * row_height
        bar_width = value / maximum * plot_width
        shown = f'{value:,.2f}'.rstrip('0').rstrip('.')
        pieces.extend([
            f'<text class="label" x="{left - 12}" y="{y + 17}" text-anchor="end">{escape(label)}</text>',
            f'<rect x="{left}" y="{y}" width="{bar_width:.2f}" height="24" rx="4" fill="{color}" opacity="0.88"/>',
            f'<text class="value" x="{min(left + bar_width + 8, width - 70):.2f}" y="{y + 17}">{shown}{escape(value_label)}</text>',
        ])
    pieces.append('</svg>')
    return ''.join(pieces)

def station_scatter_svg(frame: pd.DataFrame, title: str):
    width, height = 900, 590
    left, right, top, bottom = 75, 210, 55, 65
    plot_width, plot_height = width - left - right, height - top - bottom
    x_min, x_max = frame['longitude'].min(), frame['longitude'].max()
    y_min, y_max = frame['latitude'].min(), frame['latitude'].max()
    x_pad, y_pad = (x_max - x_min) * 0.04, (y_max - y_min) * 0.04
    x_min, x_max = x_min - x_pad, x_max + x_pad
    y_min, y_max = y_min - y_pad, y_max + y_pad

    def sx(value):
        return left + (value - x_min) / (x_max - x_min) * plot_width

    def sy(value):
        return top + (y_max - value) / (y_max - y_min) * plot_height

    provinces = sorted(frame['province'].unique())
    colors = {province: PALETTE[index % len(PALETTE)] for index, province in enumerate(provinces)}
    pieces = [
        f'<svg viewBox="0 0 {width} {height}" xmlns="http://www.w3.org/2000/svg" role="img" aria-label="{escape(title)}">',
        '<style>text{font-family:Arial,sans-serif;fill:#173042}.title{font-size:20px;font-weight:700}.axis{font-size:12px}.legend{font-size:13px}</style>',
        f'<text class="title" x="20" y="30">{escape(title)}</text>',
        f'<rect x="{left}" y="{top}" width="{plot_width}" height="{plot_height}" fill="#f8fafc" stroke="#cbd5e1"/>',
    ]
    for _, row in frame.iterrows():
        pieces.append(
            f'<circle cx="{sx(row.longitude):.2f}" cy="{sy(row.latitude):.2f}" r="3.6" fill="{colors[row.province]}" opacity="0.76"/>'
        )
    pieces.extend([
        f'<text class="axis" x="{left + plot_width / 2}" y="{height - 20}" text-anchor="middle">Longitude</text>',
        f'<text class="axis" transform="translate(20 {top + plot_height / 2}) rotate(-90)" text-anchor="middle">Latitude</text>',
    ])
    legend_x = left + plot_width + 25
    for index, province in enumerate(provinces):
        y = top + 20 + index * 34
        pieces.extend([
            f'<circle cx="{legend_x}" cy="{y}" r="6" fill="{colors[province]}"/>',
            f'<text class="legend" x="{legend_x + 13}" y="{y + 5}">{escape(province)}</text>',
        ])
    pieces.append('</svg>')
    return ''.join(pieces)

def soc_profile_svg(legs: pd.DataFrame, minimum_soc: float = 20.0):
    width, height = 900, 500
    left, right, top, bottom = 75, 210, 55, 65
    plot_width, plot_height = width - left - right, height - top - bottom
    profiles = {}
    for region, group in legs.sort_values(['region', 'sequence']).groupby('region'):
        distance = 0.0
        points = [(0.0, float(group.iloc[0]['departure_soc_percent']))]
        for _, leg in group.iterrows():
            distance += float(leg['road_distance_km'])
            points.append((distance, float(leg['arrival_soc_percent'])))
            if leg['sequence'] != group['sequence'].max():
                next_departure = float(group.loc[group['sequence'] == leg['sequence'] + 1, 'departure_soc_percent'].iloc[0])
                points.append((distance, next_departure))
        profiles[region] = points
    max_distance = max(x for points in profiles.values() for x, _ in points)

    def sx(value):
        return left + value / max_distance * plot_width

    def sy(value):
        return top + (100 - value) / 100 * plot_height

    pieces = [
        f'<svg viewBox="0 0 {width} {height}" xmlns="http://www.w3.org/2000/svg" role="img" aria-label="Profil SOC baseline feasible">',
        '<style>text{font-family:Arial,sans-serif;fill:#173042}.title{font-size:20px;font-weight:700}.axis{font-size:12px}.legend{font-size:13px}</style>',
        '<text class="title" x="20" y="30">Profil SOC pada baseline feasible</text>',
        f'<rect x="{left}" y="{top}" width="{plot_width}" height="{plot_height}" fill="#f8fafc" stroke="#cbd5e1"/>',
        f'<line x1="{left}" y1="{sy(minimum_soc):.2f}" x2="{left + plot_width}" y2="{sy(minimum_soc):.2f}" stroke="#dc2626" stroke-dasharray="7 5"/>',
        f'<text class="axis" x="{left + 8}" y="{sy(minimum_soc) - 7:.2f}" fill="#dc2626">SOC minimum {minimum_soc:.0f}%</text>',
    ]
    for index, (region, points) in enumerate(profiles.items()):
        color = PALETTE[index % len(PALETTE)]
        coordinates = ' '.join(f'{sx(x):.2f},{sy(y):.2f}' for x, y in points)
        pieces.append(f'<polyline points="{coordinates}" fill="none" stroke="{color}" stroke-width="3"/>')
        for x, y in points:
            pieces.append(f'<circle cx="{sx(x):.2f}" cy="{sy(y):.2f}" r="4" fill="{color}"/>')
        legend_y = top + 22 + index * 32
        pieces.extend([
            f'<line x1="{left + plot_width + 20}" y1="{legend_y}" x2="{left + plot_width + 47}" y2="{legend_y}" stroke="{color}" stroke-width="3"/>',
            f'<text class="legend" x="{left + plot_width + 55}" y="{legend_y + 5}">{escape(region)}</text>',
        ])
    pieces.extend([
        f'<text class="axis" x="{left + plot_width / 2}" y="{height - 20}" text-anchor="middle">Jarak jalan kumulatif (km)</text>',
        f'<text class="axis" transform="translate(20 {top + plot_height / 2}) rotate(-90)" text-anchor="middle">SOC (%)</text>',
        '</svg>',
    ])
    return ''.join(pieces)

print('Helper tabel dan visualisasi SVG siap digunakan.')

## 1. Provenance dan integritas artefak

Cell berikut memeriksa hash dataset terhadap manifest rilis dan hash setiap snapshot penelitian terhadap metadata provenance. Jika salah satu berkas berubah tanpa pembaruan metadata yang disengaja, notebook berhenti agar angka jurnal tidak tercampur dengan versi lain. Laporan JSON asli diperiksa juga apabila masih tersedia di `reports/generated/`, tetapi snapshot CSV terlacak tetap cukup untuk analisis offline setelah repository di-clone.

In [ ]:
def sha256_file(path: Path) -> str:
    return hashlib.sha256(path.read_bytes()).hexdigest()

manifest = json.loads(MANIFEST_PATH.read_text(encoding='utf-8'))
provenance = json.loads((NOTEBOOK_DATA / 'provenance.json').read_text(encoding='utf-8'))
catalog = load_station_catalog(DATASET_PATH)
catalog_summary = catalog.summary()

baseline = pd.read_csv(NOTEBOOK_DATA / 'baseline_results.csv')
sensitivity = pd.read_csv(NOTEBOOK_DATA / 'sensitivity_results.csv')
baseline_legs = pd.read_csv(NOTEBOOK_DATA / 'baseline_itinerary_legs.csv')

assert catalog.source_sha256 == manifest['dataset']['sha256']
assert catalog.source_sha256 == provenance['dataset']['sha256']
assert catalog.source_row_count == manifest['dataset']['source_rows']
assert catalog.logical_node_count == manifest['dataset']['logical_nodes']
assert manifest['application_version'] == provenance['application_version']

integrity_rows = [
    {
        'Artefak': DATASET_PATH.name,
        'Status': 'sesuai manifest',
        'SHA-256': catalog.source_sha256,
    }
]
for filename, metadata in provenance['tracked_snapshots'].items():
    path = NOTEBOOK_DATA / filename
    actual_hash = sha256_file(path)
    assert actual_hash == metadata['sha256'], filename
    frame_rows = len(pd.read_csv(path))
    assert frame_rows == metadata['rows'], filename
    integrity_rows.append({'Artefak': filename, 'Status': 'sesuai provenance', 'SHA-256': actual_hash})

for report_name, metadata in provenance['source_reports'].items():
    source_path = PROJECT_ROOT / metadata['path']
    if source_path.is_file():
        actual_hash = sha256_file(source_path)
        assert actual_hash == metadata['sha256'], report_name
        integrity_rows.append({'Artefak': metadata['path'], 'Status': 'laporan asli terverifikasi', 'SHA-256': actual_hash})

show_table(pd.DataFrame(integrity_rows), decimals=0)
print(f"Versi aplikasi: {manifest['application_version']}")
print(f"Baseline: {len(baseline)} skenario | Sensitivitas: {len(sensitivity)} skenario")

## 2. Karakteristik dataset dan cakupan spasial

Satu baris CSV dipertahankan sebagai satu unit charger. Unit yang memiliki koordinat sama sampai enam angka desimal dikonsolidasikan menjadi satu node lokasi untuk algoritma rute. Dengan demikian, informasi unit tidak hilang tetapi graf tidak memiliki dua node identik.

In [ ]:
dataset_overview = pd.DataFrame([
    {'Indikator': 'Baris/unit sumber', 'Nilai': catalog_summary['source_rows']},
    {'Indikator': 'Node lokasi logis', 'Nilai': catalog_summary['logical_nodes']},
    {'Indikator': 'Node multi-unit', 'Nilai': catalog_summary['multi_unit_node_count']},
    {'Indikator': 'Provinsi tercakup', 'Nilai': len(catalog_summary['province_counts'])},
])
show_table(dataset_overview, decimals=0)

province_counts = pd.DataFrame(
    catalog_summary['province_counts'].items(), columns=['Provinsi', 'Jumlah node']
).sort_values('Jumlah node', ascending=False)
show_table(province_counts, decimals=0)
show_svg(
    horizontal_bar_svg(province_counts['Provinsi'], province_counts['Jumlah node'], 'Distribusi node SPKLU per provinsi'),
    '01_distribusi_provinsi.svg',
)

In [ ]:
nodes_df = pd.DataFrame([node.to_dict(include_units=False) for node in catalog.nodes])
nodes_df['connectors_text'] = nodes_df['connectors'].map(', '.join)
nodes_df['networks_text'] = nodes_df['charging_networks'].map(', '.join)
show_svg(
    station_scatter_svg(nodes_df, 'Sebaran koordinat 149 node SPKLU Sulawesi'),
    '02_sebaran_node_spklu.svg',
)
print('Catatan: visualisasi ini adalah scatter koordinat, bukan peta jalan dan bukan pengganti validasi rute Google.')

In [ ]:
connector_counts = pd.DataFrame({
    'Konektor': list(CONNECTOR_ORDER),
    'Node lokasi': [catalog_summary['connector_node_counts'][name] for name in CONNECTOR_ORDER],
    'Unit sumber': [catalog_summary['connector_unit_counts'][name] for name in CONNECTOR_ORDER],
})
show_table(connector_counts, decimals=0)
show_svg(
    horizontal_bar_svg(connector_counts['Konektor'], connector_counts['Node lokasi'], 'Jumlah node menurut konektor'),
    '03_distribusi_konektor.svg',
)

network_connector = pd.DataFrame.from_dict(
    catalog_summary['network_connector_node_counts'], orient='index'
)[list(CONNECTOR_ORDER)]
network_connector.index = [CHARGING_NETWORK_LABELS[index] for index in network_connector.index]
network_connector.index.name = 'Jaringan'
display(network_connector)

### Konsolidasi unit dan pemisahan jaringan dealer

Konsolidasi dilakukan berdasarkan koordinat, sedangkan kompatibilitas tetap diperiksa pada tingkat unit. Aturan kelayakan yang diterapkan sistem adalah:

```text
unit layak = konektor unit cocok
             DAN
             (jaringan PUBLIC atau jaringan dealer dipilih pengguna)
```

Jaringan publik selalu disertakan. Checkbox Hyundai, Wuling, dan Toyota/Lexus hanya memperluas kandidat ke jaringan dealer terkait. Karena operator bisa mengubah kebijakan, pilihan tersebut tidak menyatakan kepastian akses.

In [ ]:
multi_unit_rows = []
for node in catalog.multi_unit_nodes:
    for unit in node.units:
        multi_unit_rows.append({
            'Node logis': node.name,
            'ID node': node.node_id,
            'Baris sumber': unit.source_row,
            'Nama unit': unit.name,
            'Konektor': ', '.join(unit.connectors),
        })
show_table(pd.DataFrame(multi_unit_rows), decimals=0)

selection_scenarios = [
    ('CCS2, publik saja', ('CCS2',), ()),
    ('CCS2 + checkbox Wuling', ('CCS2',), ('WULING',)),
    ('CCS2 dan GB/T, publik saja', ('CCS2', 'GB/T'), ()),
    ('CCS2 dan GB/T + checkbox Wuling', ('CCS2', 'GB/T'), ('WULING',)),
    ('AC Type 2, publik saja', ('AC TYPE 2',), ()),
    ('AC Type 2 + Hyundai + Toyota/Lexus', ('AC TYPE 2',), ('HYUNDAI', 'TOYOTA')),
]
eligibility_rows = []
for description, connectors, networks in selection_scenarios:
    eligible_nodes = [
        node for node in catalog.nodes
        if node_is_eligible(node, connectors, networks)
    ]
    eligibility_rows.append({
        'Pilihan pengguna': description,
        'Jumlah node layak': len(eligible_nodes),
    })
eligibility_df = pd.DataFrame(eligibility_rows)
show_table(eligibility_df, decimals=0)

assert eligibility_df.loc[eligibility_df['Pilihan pengguna'] == 'CCS2, publik saja', 'Jumlah node layak'].iloc[0] == 42
assert eligibility_df.loc[eligibility_df['Pilihan pengguna'] == 'CCS2 + checkbox Wuling', 'Jumlah node layak'].iloc[0] == 42
print('Interpretasi: memilih Wuling bersama CCS2 tidak menambah node dealer Wuling karena 17 node Wuling pada dataset memakai GB/T, bukan CCS2.')

## 3. Model energi dan diskretisasi SOC

Model tidak memakai kapasitas baterai atau daya charger. Energi direpresentasikan sebagai persentase State of Charge (SOC):

- `R_efektif = R_maks × alpha`
- `R_usable = ((SOC − SOC_min) / 100) × R_maks × alpha`
- `konsumsi_SOC = (jarak_jalan / (R_maks × alpha)) × 100`

Safety factor `alpha` memperkecil jangkauan nominal sebagai margin konservatif. DP membulatkan SOC kontinu **ke bawah** pada grid yang berjangkar di SOC minimum, sehingga diskretisasi tidak menambahkan energi secara semu.

In [ ]:
baseline_parameters = EnergyParameters(
    maximum_range_km=300,
    minimum_soc_percent=20,
    target_soc_percent=80,
    safety_factor=0.9,
    soc_step_percent=5,
)
current_soc = 80
discretizer = SocDiscretizer(baseline_parameters)
energy_example = pd.DataFrame([
    {'Besaran': 'Jangkauan nominal', 'Nilai': baseline_parameters.maximum_range_km, 'Satuan': 'km'},
    {'Besaran': 'Jangkauan efektif penuh', 'Nilai': baseline_parameters.effective_full_range_km, 'Satuan': 'km'},
    {'Besaran': 'Usable range pada SOC awal 80%', 'Nilai': baseline_parameters.usable_range_km(current_soc), 'Satuan': 'km'},
    {'Besaran': 'Konsumsi untuk leg 100 km', 'Nilai': baseline_parameters.consumption_percent(100), 'Satuan': '% SOC'},
    {'Besaran': 'SOC tiba setelah leg 100 km dari 80%', 'Nilai': baseline_parameters.arrival_soc_percent(80, 100), 'Satuan': '% SOC'},
])
show_table(energy_example, decimals=3)
print('Grid SOC:', discretizer.levels)
print('Contoh pembulatan konservatif: SOC 79% ->', discretizer.quantize_down(79), '%')

## 4. Pipeline rekomendasi dan Dynamic Programming

```text
Input pengguna
      │
      ▼
Rute utama Google Routes
      │
      ▼
Radius search BallTree + filter koridor/konektor/jaringan
      │
      ▼
Graf berarah maju (origin → SPKLU → destination)
      │  Haversine pruning lalu validasi jarak jalan
      ▼
DP state (node, SOC diskret)
      │  objective leksikografis
      ▼
Simulasi ulang SOC kontinu + itinerary rekomendasi
```

Urutan objective adalah waktu berkendara, jumlah berhenti, detour, SOC yang ditambahkan, lalu jarak jalan. Cell berikut memakai kelas produksi pada graf sintetis kecil untuk menunjukkan cara DP memilih pengisian, tanpa memanggil layanan eksternal.

In [ ]:
from app.services.graph import (
    DESTINATION_NODE_ID, ORIGIN_NODE_ID, GraphBuildStats, GraphEdge, GraphNode, TravelGraph
)
from app.services.optimizer import optimize_itinerary

def demo_node(node_id, kind, progress):
    names = {'origin': 'Lokasi awal', 'destination': 'Lokasi tujuan', 'station': 'SPKLU contoh'}
    return GraphNode(node_id, kind, names[kind], (0.0, progress / 100), progress)

def demo_edge(source, target, distance, duration):
    return GraphEdge(source, target, distance * 0.9, distance, duration, distance, 0.0, 250.0)

demo_graph = TravelGraph(
    nodes=(
        demo_node(ORIGIN_NODE_ID, 'origin', 0),
        demo_node('station-a', 'station', 100),
        demo_node(DESTINATION_NODE_ID, 'destination', 200),
    ),
    edges=(
        demo_edge(ORIGIN_NODE_ID, 'station-a', 100, 100),
        demo_edge('station-a', DESTINATION_NODE_ID, 100, 100),
    ),
    stats=GraphBuildStats(1, 1, 2, 0, 2, 0, 0, 0, 2, 0),
)
demo_parameters = EnergyParameters(250, 20, 80, safety_factor=1.0, soc_step_percent=5)
demo_result = optimize_itinerary(
    demo_graph, current_soc_percent=60, parameters=demo_parameters
)
assert demo_result.feasible and demo_result.itinerary is not None
demo_legs = pd.DataFrame([leg.to_dict() for leg in demo_result.itinerary.legs])
show_table(demo_legs[[
    'sequence', 'source_name', 'target_name', 'road_distance_km',
    'departure_soc_percent', 'arrival_soc_percent', 'consumption_soc_percent'
]])
show_table(pd.DataFrame([stop.to_dict() for stop in demo_result.itinerary.charging_stops]).drop(columns='station'))
print('Statistik DP:', demo_result.stats.to_dict())
print('DP mengisi dari SOC 20% menjadi 60% di SPKLU contoh agar tiba di tujuan tepat pada SOC minimum 20%.')

## 5. Hasil eksperimen baseline enam wilayah

Seluruh baseline memakai parameter yang sama: CCS2, jangkauan maksimum 300 km, SOC awal 80%, SOC minimum 20%, target SOC 80%, `alpha=0,9`, interval SOC 5%, dan radius koridor 10 km. Perbandingan antardaerah dengan parameter tetap membantu mengamati perbedaan kepadatan dan keterhubungan kandidat CCS2.

In [ ]:
baseline_feasible = baseline['route_feasible'].fillna(False).astype(bool)
baseline_aggregate = pd.DataFrame([
    {'Metrik': 'Skenario selesai', 'Nilai': int((baseline['status'] == 'completed').sum())},
    {'Metrik': 'Rute feasible', 'Nilai': int(baseline_feasible.sum())},
    {'Metrik': 'Rute infeasible', 'Nilai': int((~baseline_feasible).sum())},
    {'Metrik': 'Feasibility rate', 'Nilai': baseline_feasible.mean() * 100},
    {'Metrik': 'Pelanggaran SOC', 'Nilai': int(baseline['soc_violation_count'].sum())},
    {'Metrik': 'Rata-rata berhenti (feasible)', 'Nilai': baseline.loc[baseline_feasible, 'charging_stop_count'].mean()},
    {'Metrik': 'Total external request', 'Nilai': int(baseline['total_external_requests'].sum())},
    {'Metrik': 'Total elemen Route Matrix', 'Nilai': int(baseline['compute_route_matrix_elements'].sum())},
])
show_table(baseline_aggregate)

baseline_view = baseline[[
    'region', 'route_feasible', 'reason', 'charging_stop_count', 'charging_stop_names',
    'base_route_distance_km', 'recommended_route_distance_km',
    'minimum_observed_soc_percent', 'corridor_candidate_count', 'graph_edge_count'
]].rename(columns={
    'region': 'Wilayah', 'route_feasible': 'Feasible', 'reason': 'Alasan',
    'charging_stop_count': 'Jumlah berhenti', 'charging_stop_names': 'SPKLU terpilih',
    'base_route_distance_km': 'Jarak dasar (km)',
    'recommended_route_distance_km': 'Jarak rekomendasi (km)',
    'minimum_observed_soc_percent': 'SOC minimum teramati (%)',
    'corridor_candidate_count': 'Kandidat koridor', 'graph_edge_count': 'Edge graf',
})
show_table(baseline_view)

In [ ]:
show_svg(
    horizontal_bar_svg(baseline['region'], baseline['corridor_candidate_count'], 'Kandidat koridor pada setiap baseline'),
    '04_kandidat_baseline.svg',
)
show_svg(
    horizontal_bar_svg(baseline['region'], baseline['graph_edge_count'], 'Jumlah edge graf pada setiap baseline', color='#2563eb'),
    '05_edge_baseline.svg',
)
show_svg(
    horizontal_bar_svg(baseline['region'], baseline['compute_route_matrix_elements'], 'Elemen Route Matrix per baseline', color='#d97706'),
    '06_elemen_matrix_baseline.svg',
)

feasible_distance = baseline.loc[baseline_feasible, [
    'region', 'base_route_distance_km', 'recommended_route_distance_km', 'total_detour_km'
]].copy()
feasible_distance['perubahan_jarak_km'] = (
    feasible_distance['recommended_route_distance_km'] - feasible_distance['base_route_distance_km']
)
show_table(feasible_distance)
print('Catatan: perubahan jarak rekomendasi terhadap rute dasar tidak identik dengan total_detour_km, karena detour dicatat pada tingkat edge graf.')

In [ ]:
show_table(baseline_legs.rename(columns={
    'region': 'Wilayah', 'sequence': 'Leg', 'source_name': 'Dari', 'target_name': 'Ke',
    'road_distance_km': 'Jarak (km)', 'departure_soc_percent': 'SOC berangkat (%)',
    'arrival_soc_percent': 'SOC tiba (%)', 'consumption_soc_percent': 'Konsumsi SOC (%)',
})[[
    'Wilayah', 'Leg', 'Dari', 'Ke', 'Jarak (km)',
    'SOC berangkat (%)', 'SOC tiba (%)', 'Konsumsi SOC (%)'
]])
show_svg(soc_profile_svg(baseline_legs, minimum_soc=20), '07_profil_soc_baseline.svg')
assert baseline_legs['arrival_soc_percent'].min() >= 20
print(f"SOC tiba terendah pada seluruh leg feasible: {baseline_legs['arrival_soc_percent'].min():.3f}%")

## 6. Analisis sensitivitas satu-variabel-pada-satu-waktu

Tujuh skenario pada koridor Makassar–Rantepao mengubah satu parameter dari baseline: `alpha` (0,8; 0,9; 1,0), radius koridor (5; 10; 15 km), atau interval SOC (2,5%; 5%; 10%). Karena hanya satu koridor dan satu run per konfigurasi, hasil ini menggambarkan perilaku sistem pada kasus tersebut dan tidak boleh digeneralisasikan sebagai hukum untuk seluruh Sulawesi.

In [ ]:
sensitivity_view = sensitivity[[
    'scenario_name', 'safety_factor', 'corridor_radius_km', 'soc_step_percent',
    'charging_stop_count', 'minimum_observed_soc_percent', 'corridor_candidate_count',
    'graph_edge_count', 'dp_processed_states', 'dp_evaluated_transitions',
    'compute_route_matrix_elements', 'runtime_ms'
]].rename(columns={
    'scenario_name': 'Skenario', 'safety_factor': 'Alpha',
    'corridor_radius_km': 'Radius (km)', 'soc_step_percent': 'Interval SOC (%)',
    'charging_stop_count': 'Berhenti', 'minimum_observed_soc_percent': 'SOC minimum (%)',
    'corridor_candidate_count': 'Kandidat', 'graph_edge_count': 'Edge',
    'dp_processed_states': 'State DP', 'dp_evaluated_transitions': 'Transisi DP',
    'compute_route_matrix_elements': 'Elemen matriks', 'runtime_ms': 'Runtime (ms)',
})
show_table(sensitivity_view)

alpha_ids = ['sensitivitas-alpha-080', 'sensitivitas-baseline', 'sensitivitas-alpha-100']
corridor_ids = ['sensitivitas-koridor-5', 'sensitivitas-baseline', 'sensitivitas-koridor-15']
soc_ids = ['sensitivitas-soc-2-5', 'sensitivitas-baseline', 'sensitivitas-soc-10']
alpha_results = sensitivity.set_index('scenario_id').loc[alpha_ids].reset_index().sort_values('safety_factor')
corridor_results = sensitivity.set_index('scenario_id').loc[corridor_ids].reset_index().sort_values('corridor_radius_km')
soc_results = sensitivity.set_index('scenario_id').loc[soc_ids].reset_index().sort_values('soc_step_percent')

In [ ]:
show_svg(
    horizontal_bar_svg(
        [f"alpha={value:g}" for value in alpha_results['safety_factor']],
        alpha_results['charging_stop_count'],
        'Dampak safety factor terhadap jumlah berhenti',
    ),
    '08_sensitivitas_alpha.svg',
)
show_svg(
    horizontal_bar_svg(
        [f"radius={value:g} km" for value in corridor_results['corridor_radius_km']],
        corridor_results['compute_route_matrix_elements'],
        'Dampak radius koridor terhadap elemen Route Matrix',
        color='#d97706',
    ),
    '09_sensitivitas_radius.svg',
)
show_svg(
    horizontal_bar_svg(
        [f"interval={value:g}%" for value in soc_results['soc_step_percent']],
        soc_results['dp_evaluated_transitions'],
        'Dampak interval SOC terhadap transisi DP',
        color='#7c3aed',
    ),
    '10_sensitivitas_interval_soc.svg',
)

key_comparison = pd.DataFrame([
    {
        'Temuan': 'Alpha 0,8 vs 0,9',
        'Perubahan utama': f"berhenti {int(alpha_results.iloc[0].charging_stop_count)} vs {int(alpha_results.iloc[1].charging_stop_count)}; edge {int(alpha_results.iloc[0].graph_edge_count)} vs {int(alpha_results.iloc[1].graph_edge_count)}",
    },
    {
        'Temuan': 'Radius 5 km vs 10 km',
        'Perubahan utama': f"kandidat {int(corridor_results.iloc[0].corridor_candidate_count)} vs {int(corridor_results.iloc[1].corridor_candidate_count)}; elemen matriks {int(corridor_results.iloc[0].compute_route_matrix_elements)} vs {int(corridor_results.iloc[1].compute_route_matrix_elements)}",
    },
    {
        'Temuan': 'Interval SOC 2,5% vs 5% vs 10%',
        'Perubahan utama': 'transisi DP ' + ' vs '.join(str(int(value)) for value in soc_results['dp_evaluated_transitions']),
    },
])
show_table(key_comparison, decimals=0)

## 7. Pokok pembahasan untuk jurnal

1. **Ketersediaan lokasi tidak sama dengan keterhubungan rute.** Sulawesi Tengah, Sulawesi Tenggara, dan Sulawesi Barat memiliki kandidat koridor, tetapi graf baseline CCS2 tetap terputus di bawah usable range dan parameter yang diuji.
2. **Margin konservatif memengaruhi itinerary.** Pada Makassar–Rantepao, `alpha=0,8` mengurangi edge feasible dan menghasilkan tiga pemberhentian, sementara `alpha=0,9` dan `1,0` menghasilkan satu pemberhentian.
3. **Radius koridor memengaruhi biaya pencarian.** Radius 5 km mengurangi kandidat dan elemen matriks dibandingkan 10 km, tetapi itinerary tetap sama pada koridor uji. Radius 15 km tidak menambah kandidat dibandingkan 10 km pada snapshot ini.
4. **Interval SOC adalah trade-off ketelitian–kompleksitas.** Interval 2,5% mengevaluasi jauh lebih banyak transisi dibandingkan 5% dan 10%, sedangkan itinerary kasus uji tetap sama.
5. **Keamanan SOC dipertahankan pada solusi feasible.** Tidak terdapat pelanggaran SOC pada baseline maupun sensitivitas, dan solusi direkonstruksi lalu disimulasikan ulang menggunakan SOC kontinu.
6. **Akses dealer dipisahkan dari kompatibilitas konektor.** Checkbox dealer tidak membuat konektor yang tidak cocok menjadi cocok; contoh CCS2 + Wuling tetap tidak memasukkan unit Wuling GB/T.

In [ ]:
sensitivity_feasible = sensitivity['route_feasible'].fillna(False).astype(bool)
paper_facts = pd.DataFrame([
    {'Indikator siap dilaporkan': 'Unit sumber', 'Nilai': catalog.source_row_count, 'Konteks': 'dataset tervalidasi'},
    {'Indikator siap dilaporkan': 'Node lokasi logis', 'Nilai': catalog.logical_node_count, 'Konteks': 'setelah konsolidasi koordinat'},
    {'Indikator siap dilaporkan': 'Node CCS2', 'Nilai': catalog_summary['connector_node_counts']['CCS2'], 'Konteks': 'seluruh jaringan pada dataset'},
    {'Indikator siap dilaporkan': 'Feasibility baseline', 'Nilai': baseline_feasible.mean() * 100, 'Konteks': '6 skenario, satu per wilayah (%)'},
    {'Indikator siap dilaporkan': 'Pelanggaran SOC baseline', 'Nilai': baseline['soc_violation_count'].sum(), 'Konteks': 'seluruh leg hasil'},
    {'Indikator siap dilaporkan': 'Feasibility sensitivitas', 'Nilai': sensitivity_feasible.mean() * 100, 'Konteks': '7 skenario Makassar–Rantepao (%)'},
    {'Indikator siap dilaporkan': 'Pelanggaran SOC sensitivitas', 'Nilai': sensitivity['soc_violation_count'].sum(), 'Konteks': 'seluruh leg hasil'},
    {'Indikator siap dilaporkan': 'External request baseline', 'Nilai': baseline['total_external_requests'].sum(), 'Konteks': 'Compute Routes + request matrix'},
    {'Indikator siap dilaporkan': 'Elemen Route Matrix baseline', 'Nilai': baseline['compute_route_matrix_elements'].sum(), 'Konteks': 'bukan jumlah request'},
])
show_table(paper_facts)

assert catalog.source_row_count == 150 and catalog.logical_node_count == 149
assert int(baseline_feasible.sum()) == 3 and len(baseline) == 6
assert int(baseline['soc_violation_count'].sum()) == 0
assert sensitivity_feasible.all() and int(sensitivity['soc_violation_count'].sum()) == 0
assert int(baseline['total_external_requests'].sum()) == 40
assert int(baseline['compute_route_matrix_elements'].sum()) == 150
print('Seluruh fakta kunci konsisten dengan snapshot dan lolos assertion.')

## 8. Ancaman validitas dan batas generalisasi

- **Validitas data:** dataset tidak menyimpan status ketersediaan real-time, tarif, daya charger, antrean, jam operasional, atau kepastian akses dealer.
- **Validitas model energi:** konsumsi diasumsikan proporsional terhadap jarak dan disesuaikan dengan satu safety factor; topografi, cuaca, kecepatan, payload, serta degradasi baterai tidak dimodelkan secara eksplisit.
- **Validitas eksperimen:** baseline hanya mencakup enam koridor dan sensitivitas hanya satu koridor. Replikasi pada koridor dan waktu berbeda diperlukan untuk generalisasi yang lebih kuat.
- **Ketergantungan layanan eksternal:** jarak serta durasi jalan berasal dari respons Google Routes pada waktu eksperimen dan dapat berubah mengikuti pembaruan jaringan jalan atau layanan.
- **Runtime:** satu pengukuran per skenario tidak cukup untuk inferensi statistik performa karena latency jaringan ikut terukur.
- **Ruang lingkup objective:** waktu pengisian tidak dihitung sesuai revisi proposal; objective waktu hanya berarti waktu berkendara.

Untuk paper, sajikan keterbatasan ini secara eksplisit dan gunakan istilah **prototipe sistem pendukung perencanaan**, bukan jaminan perjalanan atau ketersediaan charger.

In [ ]:
final_validation = {
    'mode': 'offline',
    'google_api_requests_from_notebook': 0,
    'application_version': manifest['application_version'],
    'dataset_sha256_valid': catalog.source_sha256 == manifest['dataset']['sha256'],
    'baseline_rows': len(baseline),
    'sensitivity_rows': len(sensitivity),
    'baseline_error_count': int((baseline['status'] != 'completed').sum()),
    'sensitivity_error_count': int((sensitivity['status'] != 'completed').sum()),
    'charging_time_included': manifest['algorithm']['charging_time_included'],
}
assert final_validation['google_api_requests_from_notebook'] == 0
assert final_validation['dataset_sha256_valid']
assert final_validation['baseline_error_count'] == 0
assert final_validation['sensitivity_error_count'] == 0
assert final_validation['charging_time_included'] is False
display(pd.DataFrame(final_validation.items(), columns=['Pemeriksaan akhir', 'Nilai']))
print('Notebook selesai dan seluruh pemeriksaan konsistensi lulus.')

## 9. Reproduksi dan pemakaian untuk paper

1. Jalankan **Restart Kernel and Run All Cells**.
2. Pastikan seluruh assertion lulus dan cell terakhir menampilkan nol error.
3. Ubah `EXPORT_FIGURES = True` pada cell persiapan bila ingin menyimpan visualisasi SVG ke `notebooks/figures/`.
4. Gunakan `notebooks/data/provenance.json` untuk mencantumkan versi, waktu laporan, dan hash artefak.
5. Bedakan hasil empiris dari interpretasi. Angka baseline berasal dari enam skenario, sedangkan sensitivitas berasal dari satu koridor Makassar–Rantepao.
6. Jangan menjalankan ulang eksperimen live hanya untuk membuka notebook. Jika penelitian memang memerlukan pengambilan ulang, ikuti pengaman kuota dan persetujuan pada `docs/evaluation.md`.

Notebook dan snapshot ini dirancang sebagai *computational companion* untuk bagian Metode, Hasil, Pembahasan, serta Keterbatasan pada paper.